# Roster XLSX — Source Data Analysis

**File:** `User Roster EMT Only for FS 2.3.26.xlsx`  
**Purpose:** Understand employee roster data before it drives any dashboard metric.  
**Key questions:**
1. What columns does the file have?
2. What does each row represent — one employee, one certification, one role?
3. Is `Badge Number` the join key to the staffing CSV?
4. How do we identify currently active employees vs. terminated?
5. What position/rank information is available?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

FILE_PATH   = Path(r"D:\MullenAnalytics\ClientData\agencies\81d6f2e6-300a-48d0-b7b6-9250937d17fb\raw\e6532050_User Roster EMT Only for FS 2.3.26.xlsx")
STAFFING_PATH = Path(r"D:\MullenAnalytics\ClientData\agencies\81d6f2e6-300a-48d0-b7b6-9250937d17fb\raw\ec8927db_SBEMS_Staffing_2022_2025.csv")

assert FILE_PATH.exists()
print(f"File size: {FILE_PATH.stat().st_size / 1_000:.1f} KB")

## 1. Load Raw File

In [ ]:
raw = pd.read_excel(FILE_PATH)
print(f"Shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print()
print("Column names (as they appear in the file):")
for i, c in enumerate(raw.columns):
    null_n   = raw[c].isnull().sum()
    null_pct = null_n / len(raw) * 100
    print(f"  [{i}] {c!r:<30} dtype={str(raw[c].dtype):<15} nulls={null_n} ({null_pct:.0f}%)")

## 2. Random Sample — Does each row = one employee?

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 200)
raw.sample(10, random_state=42)

In [ ]:
# Confirm row = one employee by checking for duplicate badge numbers
dup_badges = raw['Badge Number'].duplicated(keep=False)
print(f"Duplicate Badge Numbers: {dup_badges.sum()} rows affected")
if dup_badges.sum() > 0:
    print("\nDuplicate rows:")
    print(raw[dup_badges].to_string())

## 3. Column Profiles

In [ ]:
# Badge Number — the join key
print("=== Badge Number ===")
print(f"  Unique values:   {raw['Badge Number'].nunique()}")
print(f"  Null count:      {raw['Badge Number'].isnull().sum()}")
print(f"  Type sample:     {raw['Badge Number'].dropna().head(5).tolist()}")
print(f"  Dtype:           {raw['Badge Number'].dtype}")
print()

# Status — active vs terminated
print("=== Status ===")
print(raw['Status'].value_counts(dropna=False).to_string())
print()

# Hire Date
print("=== Hire Date ===")
hire = pd.to_datetime(raw['Hire Date'], errors='coerce')
print(f"  Parse failures: {hire.isnull().sum() - raw['Hire Date'].isnull().sum()}")
print(f"  Range:  {hire.min().date()} → {hire.max().date()}")
print(f"  Median hire date: {hire.median().date()}")
print()

# Term Date
print("=== Term Date ===")
term = pd.to_datetime(raw['Term Date'], errors='coerce')
print(f"  Not null (terminated): {term.notna().sum()} employees ({term.notna().mean()*100:.0f}%)")
print(f"  Null (presumed active): {term.isna().sum()} employees")
print(f"  Term date range: {term.min().date() if term.notna().any() else 'N/A'} → {term.max().date() if term.notna().any() else 'N/A'}")
print()

# Skill / Rank
print("=== Skill / Rank ===")
print(raw['Skill / Rank'].value_counts(dropna=False).to_string())
print()

# User Roles
print("=== User Roles ===")
print(f"  Null: {raw['User Roles'].isnull().sum()}")
print(f"  Sample non-null:")
print(raw['User Roles'].dropna().head(5).to_string())

## 4. Active vs. Terminated employees

In [ ]:
df = raw.copy()
df['hire_date'] = pd.to_datetime(df['Hire Date'], errors='coerce')
df['term_date'] = pd.to_datetime(df['Term Date'], errors='coerce')

# Active = no term_date
df['active'] = df['term_date'].isna()

print("Active status breakdown:")
print(f"  Active (no term date):    {df['active'].sum():>4} employees")
print(f"  Terminated (has term date): {(~df['active']).sum():>4} employees")
print()
print("Active employees by Status field:")
print(df[df['active']]['Status'].value_counts().to_string())
print()
print("Terminated employees by Status field:")
print(df[~df['active']]['Status'].value_counts().to_string())
print()
# Check for inconsistency — terminated status but no term date
inconsistent = df[(df['Status'].str.lower().str.contains('term', na=False)) & df['term_date'].isna()]
print(f"Status contains 'term' but Term Date is null: {len(inconsistent)} rows")

## 5. Join Key Verification — Link to staffing CSV

In [ ]:
staffing = pd.read_csv(STAFFING_PATH, low_memory=False)

# Normalize both to integer for comparison
roster_badges  = set(df['Badge Number'].dropna().astype(int))
staffing_ids   = set(staffing['employee_id'].dropna().astype(int))

matched   = roster_badges & staffing_ids
only_rost = roster_badges - staffing_ids
only_staf = staffing_ids  - roster_badges

print(f"Roster Badge Numbers:       {len(roster_badges)}")
print(f"Staffing CSV employee_ids:  {len(staffing_ids)}")
print(f"Matched in both files:      {len(matched)}  ({len(matched)/len(roster_badges)*100:.0f}% of roster)")
print(f"In roster but not staffing: {len(only_rost)}  (employees with no shift history)")
print(f"In staffing but not roster: {len(only_staf)}  (orphan shift records)")
print()

if only_rost:
    no_shifts = df[df['Badge Number'].astype(int).isin(only_rost)][['Badge Number','Last Name','First Name','Status','Term Date']]
    print("Roster employees with no shift history (first 10):")
    print(no_shifts.head(10).to_string())

if only_staf:
    print(f"\nOrphan employee_ids in staffing CSV: {sorted(only_staf)[:10]}")

## 6. Tenure Analysis (for attrition risk context)

In [ ]:
import datetime
today = pd.Timestamp.today()

# Active employees only
active = df[df['active']].copy()
active['tenure_years'] = (today - active['hire_date']).dt.days / 365

print(f"Active employees: {len(active)}")
print(f"Tenure (years):")
print(f"  Min:    {active['tenure_years'].min():.1f}")
print(f"  Median: {active['tenure_years'].median():.1f}")
print(f"  Mean:   {active['tenure_years'].mean():.1f}")
print(f"  Max:    {active['tenure_years'].max():.1f}")
print()

# Tenure bands
bins  = [0, 1, 3, 5, 10, float('inf')]
labels = ['<1yr', '1-3yr', '3-5yr', '5-10yr', '10+yr']
active['tenure_band'] = pd.cut(active['tenure_years'], bins=bins, labels=labels)
print("Tenure distribution (active employees):")
print(active['tenure_band'].value_counts().sort_index().to_string())
print()
print("NOTE: High share of <1yr tenure suggests recent hiring wave or high turnover.")

## 7. Caveats

### 7.1 'Active' is inferred, not stated
There is no explicit 'Active' flag. An employee is treated as active if `Term Date` is null.  
An employee could be on leave, suspended, or otherwise inactive without a termination date.

### 7.2 Skill / Rank is not a standardized field
Values like 'Team Member - Driver', 'Rescue Operator - EMT' are agency-specific labels.  
These do not map directly to NFPA or state EMS certification levels without a lookup table.

### 7.3 This is a ROSTER, not a SCHEDULE
The roster tells us who is employed. It does not tell us who worked on a given day.  
For scheduling and coverage analysis, the staffing CSV (notebook 01) is the source of truth.

### 7.4 Join key is Badge Number / employee_id
This join works as long as badge numbers are not reused. If the agency reuses badge  
numbers for new hires (common in some systems), a terminated employee's shifts could  
be misattributed to their replacement.

### 7.5 Attrition risk uses this file indirectly
The forecasting module receives a staffing_summary dict, not raw roster data.  
The attrition risk model currently uses overtime_pct and gap_days from the staffing CSV.  
The roster tenure data is NOT currently fed into the attrition model — it should be.

### 7.6 What this data cannot tell you
- It cannot tell you why someone was terminated
- It cannot tell you current certification status (only 'Skill / Rank' at roster export time)
- It cannot tell you part-time availability or scheduling preferences
- It cannot tell you whether a vacant position was backfilled